# Nạp chunk vào IVFPQ index đã train

Input: `index.faiss` từ notebook `FAISS_IVFPQ_Index.ipynb` (đã train, chưa thêm vector) và `data.jsonl` gồm `doc_id`, `chunk_order`, `char_start`, `char_end`, `chunk_text`.

Output trong `/kaggle/working`: **`index.faiss`** sau khi add và **`metadata.json`**.

Bật GPU trong Kaggle Settings, sửa đường dẫn rồi chạy các cell từ trên xuống. Cần `torch`, `sentence-transformers`, `numpy`, `faiss`; notebook không tự cài thư viện. Bản `faiss-cpu` vẫn dùng được GPU để embedding, nhưng bước add chạy CPU.

## 1. Cấu hình

`start_point`, `end_point` là khoảng `doc_id`, gồm cả hai đầu. `end_point = -1` nghĩa là không giới hạn trên.

In [ ]:
import json
from pathlib import Path
from typing import TextIO

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

index_path = Path("/kaggle/input/YOUR_DATASET/index.faiss")
data_path = Path("/kaggle/input/YOUR_DATASET/data.jsonl")
output_dir = Path("/kaggle/working")

start_point = 0
end_point = -1
embedding_model = "Alibaba-NLP/gte-multilingual-base"
batch_size = 128
add_batch_size = 8_192
use_fp16 = True
use_faiss_gpu = True

output_index = output_dir / "index.faiss"
output_metadata = output_dir / "metadata.json"
output_dir.mkdir(parents=True, exist_ok=True)
if output_index.exists() or output_metadata.exists():
    raise FileExistsError("Choose another output_dir to preserve existing output files")

## 2. Đọc index và tải model

Giữ cùng model và chuẩn hóa L2 như notebook train. `use_fp16=True` dùng mixed precision khi embedding trên GPU để giảm VRAM và tăng tốc; đặt `False` để embedding bằng FP32 như lúc train.

FAISS GPU dùng bảng PQ FP16 vì index `M=64, nbits=8` cần thiết lập này theo [tài liệu FAISS GPU](https://github.com/facebookresearch/faiss/wiki/Faiss-on-the-GPU). Nếu FAISS GPU không có hoặc không chuyển được index, notebook báo lý do và dùng FAISS CPU. Mỗi lần chạy dùng GPU số 0.

In [ ]:
index = faiss.read_index(str(index_path))
if not index.is_trained or index.ntotal != 0:
    raise ValueError("Use the trained, empty index from FAISS_IVFPQ_Index.ipynb")
index.nprobe = 16

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Embedding device: {device}")
if device.type == "cpu":
    print("CUDA is unavailable; embedding runs on CPU. Enable GPU in Kaggle Settings.")
model = SentenceTransformer(embedding_model, trust_remote_code=True, device=str(device))
model.eval()

faiss_on_gpu = False
if use_faiss_gpu and device.type == "cuda" and hasattr(faiss, "StandardGpuResources"):
    try:
        gpu_resources = faiss.StandardGpuResources()
        gpu_resources.setTempMemory(256 * 1024**2)
        gpu_options = faiss.GpuClonerOptions()
        gpu_options.useFloat16 = True  # Required for 64-byte PQ codes on common Kaggle GPUs.
        index = faiss.index_cpu_to_gpu(gpu_resources, 0, index, gpu_options)
        faiss_on_gpu = True
    except RuntimeError as error:
        print(f"FAISS GPU conversion failed; using CPU: {error}")
print(f"FAISS device: {'GPU 0' if faiss_on_gpu else 'CPU'}")
if use_faiss_gpu and not faiss_on_gpu:
    print("FAISS add uses CPU because CUDA or a compatible FAISS GPU build is unavailable.")

## 3. Embedding, add vector và ghi metadata

Đọc JSONL một lần, lọc theo `doc_id`, thêm theo batch. Metadata được ghi từng batch nên RAM không tăng theo tổng số chunk.

ID vector là số nguyên liên tiếp từ 0. Các chunk cùng `doc_id` vẫn có ID riêng. `metadata.json` có dạng:

```json
{"0": {"doc_id": 2443251, "chunk_order": 0, "char_start": 0, "char_end": 220, "chunk_text": "Nội dung chunk..."}}
```

Khi search trả về ID `i`, lấy record bằng `metadata[str(i)]`. Nếu GPU hết VRAM, giảm `batch_size` xuống 64 hoặc 32; `add_batch_size` là số record gom mỗi lượt, còn `batch_size` là số text model embedding cùng lúc.

In [ ]:
def add_batch(records: list[dict], metadata_file: TextIO) -> None:
    """Embed one batch, add unique vector IDs, and write matching metadata."""
    texts = [record["chunk_text"] for record in records]
    with torch.inference_mode(), torch.autocast(
        device_type=device.type, dtype=torch.float16,
        enabled=use_fp16 and device.type == "cuda",
    ):
        vectors = model.encode(
            texts, batch_size=batch_size, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        )
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    faiss.normalize_L2(vectors)
    first_id = index.ntotal
    ids = np.arange(first_id, first_id + len(records), dtype=np.int64)
    index.add_with_ids(vectors, ids)

    if first_id:
        metadata_file.write(",")
    metadata_file.write(",".join(
        f'"{int(vector_id)}":' + json.dumps(record, ensure_ascii=False, separators=(",", ":"))
        for vector_id, record in zip(ids, records)
    ))
    print(f"\rVectors added: {index.ntotal:,}", end="", flush=True)


batch = []
with data_path.open(encoding="utf-8") as data_file, output_metadata.open("x", encoding="utf-8") as metadata_file:
    metadata_file.write("{")
    for line in data_file:
        if not line.strip():
            continue
        record = json.loads(line)
        doc_id = record["doc_id"]
        if doc_id < start_point or (end_point != -1 and doc_id > end_point):
            continue
        batch.append(record)
        if len(batch) == add_batch_size:
            add_batch(batch, metadata_file)
            batch.clear()
    if batch:
        add_batch(batch, metadata_file)
        batch.clear()
    metadata_file.write("}")
print(f"\nTotal vectors: {index.ntotal:,}")

## 4. Lưu index sau khi add

Chuyển index GPU về CPU để ghi file `.faiss`. File index đầu vào được giữ nguyên; hai output nằm trong `output_dir`. Nếu chạy lại toàn bộ, chọn thư mục output mới để giữ kết quả cũ.

In [ ]:
if output_index.exists():
    raise FileExistsError(f"Output already exists: {output_index}")
index_to_save = faiss.index_gpu_to_cpu(index) if faiss_on_gpu else index
faiss.write_index(index_to_save, str(output_index))
print(f"Index: {output_index}")
print(f"Metadata: {output_metadata}")